In [2]:
%pip install yfinance hmmlearn scikit-learn

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.1 MB 5.6 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 3.0 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 2.3 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 2.3 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.7 MB/s  0:00:01
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 2.4 MB/s eta 0:00:02
   ------- -------------------------------- 0.8/4.0 MB 1.5 MB/s eta 0:00:03
   ------- -------------------------------- 0.8/4.0 MB 1.5 MB/s eta 0:00:03
   ------------- -------------------------- 1.

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
from hmmlearn.hmm import GaussianHMM

# 1. 데이터 다운로드 (2000년~2019년 초기 학습 + 2020년~2026년 검증)
start_date = "2000-01-01"
eval_start = "2020-01-02"
end_date = "2026-11-19"

spy = yf.download("SPY", start=start_date, end=end_date, progress=False)["Close"].squeeze()
vix = yf.download("^VIX", start=start_date, end=end_date, progress=False)["Close"].squeeze()

df = pd.DataFrame({"SPY": spy, "VIX": vix}).dropna()
df["ret"] = df["SPY"].pct_change()
df["rv_21d"] = df["ret"].rolling(21).std() * np.sqrt(252) * 100
df["vol_spread"] = df["VIX"] - df["rv_21d"]
df["vix_chg"] = df["VIX"].pct_change(5)
df = df.dropna()

raw_features = ["ret", "vol_spread", "vix_chg"]

# 최근 3년(756거래일) 기준 롤링 Z-Score 표준화 (극단치 왜곡 & Look-ahead bias 동시 방지)
roll_window = 756
scaled_cols = []
for col in raw_features:
    roll_mean = df[col].rolling(roll_window, min_periods=252).mean()
    roll_std = df[col].rolling(roll_window, min_periods=252).std()
    z_col = f"{col}_z"
    df[z_col] = ((df[col] - roll_mean) / roll_std).clip(-4, 4)  # 극단 아웃라이어 클리핑
    scaled_cols.append(z_col)

df = df.dropna()


In [4]:
# 2. Statistical Jump Model (2-State, Viterbi DP 기반) 구현
def fit_predict_jump_model(X, jump_penalty=8.0, n_iter=15):
    T, D = X.shape
    centroids = np.array([X[X[:, 1] >= np.median(X[:, 1])].mean(axis=0),
                          X[X[:, 1] < np.median(X[:, 1])].mean(axis=0)])
    
    states = np.zeros(T, dtype=int)
    for _ in range(n_iter):
        loss_matrix = np.zeros((T, 2))
        for k in range(2):
            loss_matrix[:, k] = np.sum((X - centroids[k]) ** 2, axis=1)
        
        dp = np.zeros((T, 2))
        backptr = np.zeros((T, 2), dtype=int)
        dp[0] = loss_matrix[0]
        
        for t in range(1, T):
            for k in range(2):
                stay_cost = dp[t-1, k]
                switch_cost = dp[t-1, 1-k] + jump_penalty
                if stay_cost <= switch_cost:
                    dp[t, k] = stay_cost + loss_matrix[t, k]
                    backptr[t, k] = k
                else:
                    dp[t, k] = switch_cost + loss_matrix[t, k]
                    backptr[t, k] = 1 - k
        
        states[-1] = np.argmin(dp[-1])
        for t in range(T - 2, -1, -1):
            states[t] = backptr[t + 1, states[t + 1]]
            
        for k in range(2):
            if np.sum(states == k) > 0:
                centroids[k] = X[states == k].mean(axis=0)
                
    if centroids[0, 0] < centroids[1, 0]:
        states = 1 - states
    return states


In [8]:
# 3. Walk-Forward (Expanding Window) 실시간 레짐 탐지
eval_idx = df.index[df.index >= eval_start]
hmm_regimes = []
jm_regimes = []

retrain_step = 21  # 매월(21거래일) 재학습
current_hmm = None

for i, date in enumerate(eval_idx):
    X_scaled = df.loc[:date, scaled_cols].values
    
    if i % retrain_step == 0:
        hmm = GaussianHMM(n_components=2, covariance_type="full", n_iter=100, random_state=42)
        hmm.fit(X_scaled)
        hmm_crisis_state = 0 if hmm.means_[0, 0] < hmm.means_[1, 0] else 1
        current_hmm = (hmm, hmm_crisis_state)
        
        jm_states = fit_predict_jump_model(X_scaled[-1500:], jump_penalty=30.0)
        jm_regimes.append(jm_states[-1])
    else:
        jm_states = fit_predict_jump_model(X_scaled[-504:], jump_penalty=30.0)
        jm_regimes.append(jm_states[-1])
        
    hmm_model, crisis_idx = current_hmm
    post_probs = hmm_model.predict_proba(X_scaled[-504:])
    hmm_regimes.append(1 if post_probs[-1, crisis_idx] > 0.5 else 0)

res = df.loc[eval_idx].copy()
res["HMM_Crisis"] = hmm_regimes
res["JM_Crisis"] = jm_regimes



In [10]:
# 4. EWMA 변동성 스케일링 (lambda = 0.94)
res["ewma_vol"] = res["ret"].ewm(alpha=0.06).std() * np.sqrt(252)
target_vol = 0.16
res["vol_scalar"] = (target_vol / res["ewma_vol"].shift(1)).clip(upper=1.0).fillna(1.0)

res["HMM_weight"] = np.where(res["HMM_Crisis"].shift(1) == 1, 0.5, 1.0) * res["vol_scalar"]
res["JM_weight"] = np.where(res["JM_Crisis"].shift(1) == 1, 0.5, 1.0) * res["vol_scalar"]

cost_bps = 0.0005
res["SPY_ret"] = res["ret"]
res["HMM_ret"] = res["HMM_weight"] * res["ret"] - res["HMM_weight"].diff().abs().fillna(0) * cost_bps
res["JM_ret"] = res["JM_weight"] * res["ret"] - res["JM_weight"].diff().abs().fillna(0) * cost_bps


In [12]:
# 5. 성과 요약 테이블 출력
def calc_metrics(r, switches=None):
    cum = (1 + r).cumprod()
    tot_ret = (cum.iloc[-1] - 1) * 100
    cagr = (cum.iloc[-1] ** (252 / len(r)) - 1) * 100
    sharpe = (r.mean() / r.std()) * np.sqrt(252)
    mdd = ((cum / cum.cummax()) - 1).min() * 100
    return [f"{tot_ret:.2f}%", f"{cagr:.2f}%", f"{sharpe:.2f}", f"{mdd:.2f}%", switches if switches is not None else "—"]

hmm_switches = int(res["HMM_Crisis"].diff().abs().sum())
jm_switches = int(res["JM_Crisis"].diff().abs().sum())

summary = pd.DataFrame({
    "S&P 500 (SPY)": calc_metrics(res["SPY_ret"]),
    "Gaussian HMM + EWMA": calc_metrics(res["HMM_ret"], hmm_switches),
    "Jump Model + EWMA": calc_metrics(res["JM_ret"], jm_switches)
}, index=["Total Return", "CAGR", "Sharpe Ratio", "Max Drawdown", "Regime Switches"])

print("\n=== Phase 2 Prototype: Regime Detection + EWMA Vol Scaling (2020-01-02 to 2026-11-18) ===")
print(summary.to_string())


=== Phase 2 Prototype: Regime Detection + EWMA Vol Scaling (2020-01-02 to 2026-11-18) ===
                S&P 500 (SPY) Gaussian HMM + EWMA Jump Model + EWMA
Total Return          149.51%             103.49%            61.83%
CAGR                   14.25%              10.90%             7.26%
Sharpe Ratio             0.77                0.92              0.86
Max Drawdown          -33.72%             -15.90%           -11.37%
Regime Switches             —                  93                52
